# 02 数据清洗与指标口径

## 本步目标

1. 检查关键字段缺失、重复和状态分布。
2. 确认日期映射和金额归一化是否正确。
3. 把 5 个核心指标口径写成可复用的查询。

## 清洗原则

- 状态为空：保留，但在报告中标注“未知”，不擅自归入通过或拒绝。
- `Canceled`：客户取消或流程取消，保留在进件分母。
- `Unused offer`：审批通过但未用信，计入通过、不计入用信。
- 金额：统一除以 10,000，单位万元。
- 日期：`DAYS_*` 是负数，映射时使用 `DATE_ADD(锚点, INTERVAL DAYS_* DAY)`。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 关键字段缺失率

In [2]:
display(query_df('''
SELECT
    ROUND(SUM(status IS NULL OR status = '') / COUNT(*), 4) AS 状态缺失率,
    ROUND(SUM(channel_type IS NULL OR channel_type = '') / COUNT(*), 4) AS 渠道缺失率,
    ROUND(SUM(approved_amount IS NULL) / COUNT(*), 4) AS 审批金额缺失率,
    ROUND(SUM(decision_date IS NULL) / COUNT(*), 4) AS 日期缺失率
FROM `v_申请明细`
'''))

,状态缺失率,渠道缺失率,审批金额缺失率,日期缺失率
0,0.0,0.0,0.0,0.0


## 2. 主键与日期范围校验

In [3]:
display(query_df("SELECT COUNT(*) - COUNT(DISTINCT previous_id) AS 申请主键重复数 FROM `v_申请明细`"))
display(query_df("SELECT MIN(decision_date) AS 最早, MAX(decision_date) AS 最晚 FROM `v_申请明细`"))

,申请主键重复数
0,0


,最早,最晚
0,2018-09-30,2026-09-29


## 3. 最近两个月核心指标

In [4]:
kpi = query_df('''
SELECT * FROM `v_申请审批月度`
WHERE `月份` IN ('2026-08','2026-09')
ORDER BY `月份`
''')
display(kpi)

,月份,进件量,审批通过量,实际用信量,拒绝量,取消量,未使用额度量,审批通过率,用信率,审批金额,件均
0,2026-08,28537,8394.0,7975.0,4292.0,15851.0,419.0,0.2941,0.9501,142868.64,17.91
1,2026-09,36012,8245.0,8070.0,6827.0,20940.0,175.0,0.2290,0.9788,188680.95,23.38


## 4. 用 Pandas 复算一次通过率，确认没有对“比率”求平均

In [5]:
kpi["手工通过率"] = kpi["审批通过量"] / kpi["进件量"]
kpi["手工用信率"] = kpi["实际用信量"] / kpi["审批通过量"]
display(kpi[["月份", "进件量", "审批通过量", "实际用信量", "审批通过率", "手工通过率", "用信率", "手工用信率"]])

,月份,进件量,审批通过量,实际用信量,审批通过率,手工通过率,用信率,手工用信率
0,2026-08,28537,8394.0,7975.0,0.2941,0.294144,0.9501,0.950083
1,2026-09,36012,8245.0,8070.0,0.2290,0.228951,0.9788,0.978775


## 本步结论

数据质量检查通过：申请主键无重复，关键字段缺失可控；2026-09 审批通过率 22.90%、用信率 97.88%，与 SQL 视图一致。